In [1]:
import nltk
nltk.download('reuters')
from nltk.corpus import reuters

[nltk_data] Downloading package reuters to /Users/pb/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


In [2]:
fileids = reuters.fileids()

print(reuters.raw(fileids[0]))

ASIAN EXPORTERS FEAR DAMAGE FROM U.S.-JAPAN RIFT
  Mounting trade friction between the
  U.S. And Japan has raised fears among many of Asia's exporting
  nations that the row could inflict far-reaching economic
  damage, businessmen and officials said.
      They told Reuter correspondents in Asian capitals a U.S.
  Move against Japan might boost protectionist sentiment in the
  U.S. And lead to curbs on American imports of their products.
      But some exporters said that while the conflict would hurt
  them in the long-run, in the short-term Tokyo's loss might be
  their gain.
      The U.S. Has said it will impose 300 mln dlrs of tariffs on
  imports of Japanese electronics goods on April 17, in
  retaliation for Japan's alleged failure to stick to a pact not
  to sell semiconductors on world markets at below cost.
      Unofficial Japanese estimates put the impact of the tariffs
  at 10 billion dlrs and spokesmen for major electronics firms
  said they would virtually halt exports

In [3]:
reuters.categories(fileids[0])

['trade']

In [4]:
len(fileids)

10788

# 2. Data Preprocessing 


In [18]:
text_labels = [(reuters.raw(fid).lower(), reuters.categories(fid)) for fid in fileids]
text_labels[0]

('asian exporters fear damage from u.s.-japan rift\n  mounting trade friction between the\n  u.s. and japan has raised fears among many of asia\'s exporting\n  nations that the row could inflict far-reaching economic\n  damage, businessmen and officials said.\n      they told reuter correspondents in asian capitals a u.s.\n  move against japan might boost protectionist sentiment in the\n  u.s. and lead to curbs on american imports of their products.\n      but some exporters said that while the conflict would hurt\n  them in the long-run, in the short-term tokyo\'s loss might be\n  their gain.\n      the u.s. has said it will impose 300 mln dlrs of tariffs on\n  imports of japanese electronics goods on april 17, in\n  retaliation for japan\'s alleged failure to stick to a pact not\n  to sell semiconductors on world markets at below cost.\n      unofficial japanese estimates put the impact of the tariffs\n  at 10 billion dlrs and spokesmen for major electronics firms\n  said they would 

In [19]:
from nltk import word_tokenize

for i, elem in enumerate(text_labels):
    text_labels[i] = (word_tokenize(elem[0]), elem[1])

In [21]:
from nltk import WordNetLemmatizer

wnl = WordNetLemmatizer()

text_labels = [([wnl.lemmatize(token) for token in tokens], labels)
               for tokens, labels in text_labels]

In [29]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

for i, (tokens, labels) in enumerate(text_labels):
    text_labels[i] = ([token for token in tokens if token not in stop_words], labels)

# 2. Data Preprocessing with Spark


In [ ]:
# text_labels = [(reuters.raw(fid).lower(), reuters.categories(fid)) for fid in fileids]

In [ ]:
# from pyspark import SparkContext
# sc = SparkContext("local", "example")

In [ ]:
# fileids = reuters.fileids()
# text_labels = [(reuters.raw(fid), reuters.categories(fid)) for fid in fileids]
# rdd1 = sc.parallelize(text_labels)

In [ ]:
# text_labels = rdd1.map(lambda x: (x[0].lower(), x[1]))
# text_labels.take(3)

In [ ]:
# from nltk import word_tokenize

# text_labels = text_labels.map(lambda x: (word_tokenize(x[0]), x[1]))
# text_labels.take(3)

In [ ]:
# def lemmatize_partition(it):
#     from nltk.stem import WordNetLemmatizer
#     wnl = WordNetLemmatizer()
#     for tokens, labels in it:                 # tokens is a list[str]
#         yield ([wnl.lemmatize(t) for t in tokens], labels)

# text_labels = text_labels.mapPartitions(lemmatize_partition)
# text_labels.take(3)

# 3. Document Classification - TF-IDF

In [96]:
texts = [' '.join(text) for (text, labels) in text_labels]
labels = [1 if 'earn' in label else 0 for (text, label) in text_labels]

texts[0]

"asian exporter fear damage u.s.-japan rift mounting trade friction u.s. japan ha raised fear among many asia 's exporting nation row could inflict far-reaching economic damage , businessmen official said . told reuter correspondent asian capital u.s. move japan might boost protectionist sentiment u.s. lead curb american import product . exporter said conflict would hurt long-run , short-term tokyo 's loss might gain . u.s. ha said impose 300 mln dlrs tariff import japanese electronics good april 17 , retaliation japan 's alleged failure stick pact sell semiconductor world market cost . unofficial japanese estimate put impact tariff 10 billion dlrs spokesman major electronics firm said would virtually halt export product hit new tax . `` would n't able business , '' said spokesman leading japanese electronics firm matsushita electric industrial co ltd & lt ; mc.t > . `` tariff remain place length time beyond month mean complete erosion export ( good subject tariff ) u.s. , '' said tom 

In [97]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

In [98]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_matrix = vectorizer.fit_transform(X_train)

In [99]:
import pandas as pd

def print_matrix(X, vectorizer):
    dense_matrix = X.todense()
    feature_names = vectorizer.get_feature_names_out()
    df = pd.DataFrame(dense_matrix, columns=feature_names)
    print (df)

print_matrix(X_matrix, vectorizer)

       00       000  0000  00000  0009  001  002       003  0037  004  ...  \
0     0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
1     0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
2     0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
3     0.0  0.153875   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
4     0.0  0.022909   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
...   ...       ...   ...    ...   ...  ...  ...       ...   ...  ...  ...   
8625  0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
8626  0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.268608   0.0  0.0  ...   
8627  0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
8628  0.0  0.000000   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   
8629  0.0  0.409571   0.0    0.0   0.0  0.0  0.0  0.000000   0.0  0.0  ...   

      zuckerman  zuheir  zulia  zurich  zuyuan  zverev  zwerman

In [100]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier()
clf.fit(X_matrix, y_train)

DecisionTreeClassifier()

In [101]:
import numpy as np

# Get the feature importances
importances = clf.feature_importances_

# Get the feature (tern) names
feature_names = vectorizer.get_feature_names_out()

# Sort the features by importance
indices = np.argsort(importances)[::-1]

# Print the top n most discriminant terms (e.g., top 10)
top_n = 10
for i in range(top_n):
    print(f"(i+1). Feature: {feature_names[indices[i]]}, Importance: {importances[indices[i]]}")

(i+1). Feature: ct, Importance: 0.5773835799178961
(i+1). Feature: net, Importance: 0.11251141476642718
(i+1). Feature: profit, Importance: 0.03925706991689244
(i+1). Feature: price, Importance: 0.03534950957212952
(i+1). Feature: split, Importance: 0.029840799603544597
(i+1). Feature: dividend, Importance: 0.018406503026490356
(i+1). Feature: lt, Importance: 0.015089502034301218
(i+1). Feature: earnings, Importance: 0.011873843050708583
(i+1). Feature: loss, Importance: 0.011250871536708503
(i+1). Feature: 1987, Importance: 0.00685717799534987


In [102]:
from sklearn.metrics import accuracy_score

test_X_matrix = vectorizer.transform(X_test)
y_pred = clf.predict(test_X_matrix)
accuracy = accuracy_score(y_test, y_pred)

print(accuracy)

0.958758109360519


In [103]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_matrix, y_train)
model.score(test_X_matrix, y_test)

0.9763670064874884

# 4. Document Classification - Word Embeddings

In [63]:
import gensim
from gensim.downloader import load

model = load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [64]:
embedding = model['profit']
embedding

array([ 0.44504  , -0.83999  ,  1.251    ,  0.0077679,  0.41529  ,
        0.075351 , -0.2221   , -0.93953  ,  0.65541  ,  0.93378  ,
        0.056446 , -0.32597  , -0.95686  , -1.0279   ,  0.065953 ,
       -0.2115   , -0.61723  , -0.45928  , -0.83765  , -0.58279  ,
        1.0887   , -0.42576  , -0.1395   , -0.98605  , -0.69112  ,
       -0.57973  , -1.1405   , -0.28078  ,  0.10668  ,  0.5845   ,
        3.1997   ,  1.265    ,  1.6451   ,  1.25     , -0.24423  ,
       -1.5106   , -0.59652  ,  0.054633 ,  0.59343  , -1.0555   ,
       -0.19028  , -0.6877   ,  0.40265  ,  0.041532 , -0.26277  ,
       -0.20903  , -0.41595  ,  0.79415  ,  0.83228  ,  0.49215  ],
      dtype=float32)

In [79]:
similar = model.most_similar('profit', topn=10)
similar

[('earnings', 0.9254103899002075),
 ('profits', 0.9171135425567627),
 ('shares', 0.8507129549980164),
 ('sales', 0.847875714302063),
 ('revenue', 0.8461629152297974),
 ('net', 0.8377792239189148),
 ('share', 0.8344246745109558),
 ('pretax', 0.8288366198539734),
 ('gains', 0.8283697962760925),
 ('revenues', 0.8163827061653137)]

In [87]:
def naive(doc):
    embeddings = []

    for word in doc:
        if word in model:
            embeddings.append(model[word])

    avg_embedding = np.mean(embeddings, axis=0)

    return avg_embedding

result = naive(['text', 'two'])
print(result)

[ 0.45452     0.36472     0.16657975 -0.005685    0.50526     0.505165
 -0.567105   -0.49289003 -0.52613    -0.22474998 -0.29999     0.01905
 -0.09949002  0.379435    0.2696     -0.448475   -0.569007   -0.835165
 -0.04743502 -0.39304     0.1444805  -0.13306499  0.55992496  0.29643852
  0.32365498 -0.976815   -0.08416499 -0.27656    -0.42232    -0.701235
  3.50225    -0.403485   -0.274024   -0.162467    0.46635     0.2280565
  0.45523     0.19688    -0.762765    0.274175    0.54156    -0.00986
  0.11781999  0.073265   -0.08507325  0.457827    0.375785    0.39054
 -0.24987    -0.71832   ]


In [91]:
train_doc_embeddings = []
for doc in X_train:
    train_doc_embeddings.append(naive(doc))

test_doc_embeddings = []
for doc in X_test:
    test_doc_embeddings.append(naive(doc))

In [92]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(train_doc_embeddings, y_train)
model.score(test_doc_embeddings, y_test)

0.8878591288229842

In [94]:
from sklearn.svm import SVC

model = SVC()
model.fit(train_doc_embeddings, y_train)
model.score(test_doc_embeddings, y_test)

0.8795180722891566

## Comparison

| Method | Feature Representation | Accuracy |
|--------|----------------------|----------|
| Decision Tree | TF-IDF | 96.2% |
| Logistic Regression | TF-IDF | 97.6% |
| Logistic Regression | Word Embeddings | 88.8% |
| SVM | Word Embeddings | 88.0% |

TF-IDF outperforms word embeddings, with a ~9% difference. Logistic regression is the strongest performer with 97.6% accuracy. 

The feature representation has a larger impact on accuracy then the choice of algorithm. 

TF-IDF is the more interpretable representation as we can see the contribution of each feature to decision. 